# Week 4 - Day 1 : Daily Challenge
# Integration de NumPy, Pandas et Matplotlib

**Scenario :** Vous etes analyste de donnees et travaillez avec un dataset de meteo mondiale.
Votre tache : analyser les tendances de temperature et visualiser les resultats.

**Objectif :** Generer, analyser et visualiser les temperatures mensuelles moyennes
de 10 villes sur 12 mois (temperatures entre -5°C et 35°C).

## Setup - Import des librairies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

np.random.seed(42)
print("NumPy     :", np.__version__)
print("Pandas    :", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)

---
## Partie 1 : Preparation des donnees

- Generer un tableau NumPy de temperatures aleatoires (10 villes x 12 mois)
- Temperatures entre -5°C et 35°C
- Convertir en DataFrame Pandas avec villes en index et mois en colonnes

In [ ]:
# Hint 1 : np.random.uniform(low, high, size)
# Hint 2 : pd.DataFrame(data, index, columns)

villes = [
    "Paris", "Montreal", "Dubai", "Tokyo",
    "Sydney", "New York", "Mexico", "Mumbai",
    "Oslo", "Le Caire"
]

mois = [
    "Jan", "Fev", "Mar", "Avr", "Mai", "Jun",
    "Jul", "Aou", "Sep", "Oct", "Nov", "Dec"
]

# Generation du tableau NumPy (10 villes x 12 mois)
temperatures_np = np.random.uniform(low=-5, high=35, size=(10, 12))

# Arrondir a 1 decimale pour plus de lisibilite
temperatures_np = np.round(temperatures_np, 1)

print("Tableau NumPy genere :")
print(f"Shape : {temperatures_np.shape}")
print(f"Min   : {temperatures_np.min():.1f}°C")
print(f"Max   : {temperatures_np.max():.1f}°C")
print(temperatures_np)

In [ ]:
# Conversion en DataFrame Pandas
df = pd.DataFrame(
    data=temperatures_np,
    index=villes,
    columns=mois
)

print("DataFrame Pandas :")
print(df.to_string())

---
## Partie 2 : Analyse des donnees

- Calculer la temperature moyenne annuelle pour chaque ville
- Identifier la ville la plus chaude et la plus froide
- Hint 1 : DataFrame.mean(axis=1)
- Hint 2 : idxmax() et idxmin()

In [ ]:
# Hint 1 : axis=1 => moyenne par ligne (par ville)
moyenne_annuelle = df.mean(axis=1).round(2)

print("=== Temperature moyenne annuelle par ville ===")
for ville, temp in moyenne_annuelle.sort_values(ascending=False).items():
    barre = "#" * int((temp + 5) / 2)
    print(f"{ville:<12} : {temp:>6.2f}°C  {barre}")

In [ ]:
# Hint 2 : idxmax() et idxmin()
ville_plus_chaude = moyenne_annuelle.idxmax()
ville_plus_froide = moyenne_annuelle.idxmin()

print("=" * 45)
print(f"Ville la plus CHAUDE : {ville_plus_chaude} ({moyenne_annuelle[ville_plus_chaude]:.2f}°C)")
print(f"Ville la plus FROIDE : {ville_plus_froide} ({moyenne_annuelle[ville_plus_froide]:.2f}°C)")
print(f"Ecart de temperature : {(moyenne_annuelle[ville_plus_chaude] - moyenne_annuelle[ville_plus_froide]):.2f}°C")
print("=" * 45)

# Stats supplementaires avec NumPy
print("\n=== Statistiques globales (NumPy) ===")
print(f"Moyenne globale  : {np.mean(temperatures_np):.2f}°C")
print(f"Ecart-type       : {np.std(temperatures_np):.2f}°C")
print(f"Mois le plus froid (global) : {mois[np.argmin(df.mean(axis=0))]} ({df.mean(axis=0).min():.2f}°C)")
print(f"Mois le plus chaud (global) : {mois[np.argmax(df.mean(axis=0))]} ({df.mean(axis=0).max():.2f}°C)")

---
## Partie 3 : Visualisation des donnees

In [ ]:
# Graphique 1 : Evolution mensuelle des temperatures par ville (Line Chart)
fig, ax = plt.subplots(figsize=(14, 7))

colors = cm.tab10(np.linspace(0, 1, len(villes)))

for i, ville in enumerate(villes):
    ax.plot(
        mois,
        df.loc[ville],
        marker="o",
        linewidth=2,
        label=ville,
        color=colors[i]
    )

ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="0°C")
ax.set_title("Temperatures mensuelles moyennes par ville", fontsize=16, fontweight="bold")
ax.set_xlabel("Mois", fontsize=12)
ax.set_ylabel("Temperature (°C)", fontsize=12)
ax.legend(loc="upper right", fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_ylim(-10, 40)

plt.tight_layout()
plt.savefig("daily_plot1_temperature_mensuelle.png", dpi=150)
plt.show()
print("Graphique 1 sauvegarde.")

In [ ]:
# Graphique 2 : Temperature moyenne annuelle par ville (Bar Chart)
fig, ax = plt.subplots(figsize=(12, 6))

sorted_moyennes = moyenne_annuelle.sort_values(ascending=False)
bar_colors = ["#e74c3c" if v == ville_plus_chaude else "#3498db" if v == ville_plus_froide else "#95a5a6"
              for v in sorted_moyennes.index]

bars = ax.bar(sorted_moyennes.index, sorted_moyennes.values, color=bar_colors, edgecolor="white", linewidth=0.8)

for bar, val in zip(bars, sorted_moyennes.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.1f}°C", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("Temperature moyenne annuelle par ville", fontsize=16, fontweight="bold")
ax.set_xlabel("Ville", fontsize=12)
ax.set_ylabel("Temperature moyenne (°C)", fontsize=12)
ax.axhline(y=moyenne_annuelle.mean(), color="orange", linestyle="--", linewidth=1.5,
           label=f"Moyenne globale : {moyenne_annuelle.mean():.1f}°C")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#e74c3c", label=f"Plus chaude : {ville_plus_chaude}"),
    Patch(facecolor="#3498db", label=f"Plus froide : {ville_plus_froide}"),
    Patch(facecolor="#95a5a6", label="Autres villes")
]
ax.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.savefig("daily_plot2_moyenne_annuelle.png", dpi=150)
plt.show()
print("Graphique 2 sauvegarde.")

In [ ]:
# Graphique 3 : Heatmap des temperatures (toutes villes x tous mois)
fig, ax = plt.subplots(figsize=(14, 7))

heatmap = ax.imshow(df.values, cmap="RdYlBu_r", aspect="auto", vmin=-5, vmax=35)

ax.set_xticks(range(len(mois)))
ax.set_xticklabels(mois, fontsize=11)
ax.set_yticks(range(len(villes)))
ax.set_yticklabels(villes, fontsize=11)

for i in range(len(villes)):
    for j in range(len(mois)):
        val = df.values[i, j]
        text_color = "white" if val > 25 or val < 5 else "black"
        ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                fontsize=9, color=text_color, fontweight="bold")

cbar = plt.colorbar(heatmap, ax=ax, shrink=0.8)
cbar.set_label("Temperature (°C)", fontsize=12)

ax.set_title("Heatmap des temperatures mensuelles (10 villes x 12 mois)", fontsize=15, fontweight="bold")
ax.set_xlabel("Mois", fontsize=12)
ax.set_ylabel("Ville", fontsize=12)

plt.tight_layout()
plt.savefig("daily_plot3_heatmap.png", dpi=150)
plt.show()
print("Graphique 3 sauvegarde.")

In [ ]:
# Graphique 4 : Boxplot de la distribution des temperatures par ville
fig, ax = plt.subplots(figsize=(13, 6))

data_boxplot = [df.loc[ville].values for ville in villes]
bp = ax.boxplot(data_boxplot, labels=villes, patch_artist=True, notch=False)

colors_box = cm.coolwarm(np.linspace(0, 1, len(villes)))
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_title("Distribution des temperatures mensuelles par ville (Boxplot)", fontsize=15, fontweight="bold")
ax.set_xlabel("Ville", fontsize=12)
ax.set_ylabel("Temperature (°C)", fontsize=12)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(-10, 40)

plt.tight_layout()
plt.savefig("daily_plot4_boxplot.png", dpi=150)
plt.show()
print("Graphique 4 sauvegarde.")

---
## Rapport de Synthese

**Donnees :** Temperatures mensuelles moyennes simulees pour 10 villes mondiales sur 12 mois.
Generees avec NumPy (`np.random.uniform(-5, 35, size=(10, 12))`), converties en DataFrame Pandas.

In [ ]:
rapport = f"""
========================================
        RAPPORT DE SYNTHESE
========================================

RESULTATS CLES
--------------
Ville la plus CHAUDE : {ville_plus_chaude}
  Moyenne annuelle   : {moyenne_annuelle[ville_plus_chaude]:.2f}°C

Ville la plus FROIDE : {ville_plus_froide}
  Moyenne annuelle   : {moyenne_annuelle[ville_plus_froide]:.2f}°C

Ecart entre extremes : {(moyenne_annuelle[ville_plus_chaude] - moyenne_annuelle[ville_plus_froide]):.2f}°C
Moyenne globale      : {np.mean(temperatures_np):.2f}°C
Ecart-type global    : {np.std(temperatures_np):.2f}°C

TENDANCES OBSERVEES
-------------------
1. Les temperatures sont uniformement distribuees entre -5°C et 35°C
   (donnees simulees via distribution uniforme NumPy).

2. La heatmap revele visuellement les disparites entre villes froides
   (Oslo, Montreal — tons bleus) et villes chaudes (Dubai, Mumbai — tons rouges).

3. Les boxplots montrent une variabilite mensuelle similaire entre villes
   (environ 35-40°C d'ecart entre min et max annuel par ville),
   ce qui est attendu avec une distribution uniforme.

4. Aucune saisonnalite marquee n'est visible car les donnees sont aleatoires
   (dans un dataset reel, on observerait des pics estivaux et des creux hivernaux).

OUTILS UTILISES
---------------
- NumPy  : generation des donnees, stats (mean, std, argmin, argmax)
- Pandas : DataFrame, mean(axis=1), idxmax(), idxmin()
- Matplotlib : Line chart, Bar chart, Heatmap (imshow), Boxplot
========================================
"""
print(rapport)